In [1]:
import pandas as pd
import numpy as np
import re

#ploting
import matplotlib.pyplot as plt
import seaborn as sns

#train_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import json

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding,Bidirectional,LSTM,Dense,Dropout
from tensorflow.keras.layers import Input, GlobalAveragePooling1D, Attention
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC
from tensorflow.keras.models import Model



#Reading csv file


In [ ]:
import pandas as pd

# Load spam dataset
spam = pd.read_csv("spam.csv", encoding="latin1")
spam = spam[['v1','v2']]              # pick only relevant columns
spam.columns = ['label','text']       # rename

# Load feedback dataset
feedback = pd.read_csv("feedback.csv", encoding="latin1")
feedback = feedback[['label','message']]  # pick relevant columns
feedback.columns = ['label','text']       # rename to match spam

# Concatenate
df = pd.concat([spam, feedback], ignore_index=True)

# Ensure label column is string
df['label'] = df['label'].astype(str)

# Lowercase + map to 0/1
df['label'] = df['label'].str.lower().map({'ham':0,'spam':1})

# Drop rows with NaN label
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(df.head())

   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [ ]:
df.shape

(5611, 2)

#finding missing values


In [ ]:
# # Rename both columns correctly
# df = df.rename(columns={'v1':'label', 'v2':'text'})

# # Keep only label + text
# df = df[['label', 'text']]

# # Ensure all labels are strings first
# df['label'] = df['label'].astype(str)

# # Convert to lowercase and map to 0/1
# df['label'] = df['label'].str.lower().map({'ham':0,'spam':1})

# # Drop rows where label becamae NaN (invalid/missing)
# df = df.dropna(subset=['label'])

# # Convert to integer type
# df['label'] = df['label'].astype(int)

# print(df.head())
# print(df['label'].unique())  # should show only [0,1]


In [ ]:
#clean and preprocess text (removing emoji and converting unnecessary things and to lowercase)
def clean_text(text):
    text=text.lower()
    text=re.sub(r"http\S+","",text)
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text


df['cleaned_text']=df['text'].apply(clean_text)

df.head()


,label,text,cleaned_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


In [ ]:
tokenizer=Tokenizer(num_words=5000,oov_token='<OOV>')
#Creates a Tokenizer object that converts words to numbers.
#It will keep only the top 10,000 most frequent words.
#Any unknown (rare) word will be replaced with the token <OOV> (out-of-vocabulary).

tokenizer.fit_on_texts(df['cleaned_text'])
#Goes through all your cleaned text and builds a word index (a dictionary of word → ID).

sequences=tokenizer.texts_to_sequences(df['cleaned_text'])
#Converts each sentence into a list of word IDs.

padded=pad_sequences(sequences,maxlen=100,padding='post',truncating='post')
#Ensures all sequences are exactly 100 tokens long (Bi-LSTM expects same-length inputs).
#If a sentence is:
#Shorter than 100 → it adds 0s at the end (post-padding)
#Longer than 100 → it cuts off extra words at the end (post-truncating)



#Splitting data for training and testing

In [ ]:

X=padded
y=df['label'].values

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#Building Bi-LSTM model

In [6]:
inputs = Input(shape=(100,))
embedding = Embedding(input_dim=5000, output_dim=128)(inputs)
bilstm = Bidirectional(LSTM(64, return_sequences=True))(embedding)
attention = Attention()([bilstm, bilstm])
pool = GlobalAveragePooling1D()(attention)
drop = Dropout(0.5)(pool)
dense = Dense(32, activation='relu')(drop)
output = Dense(1, activation='sigmoid')(dense)
model = Model(inputs, output)

In [7]:
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
#Embedding(input_dim=10000, output_dim=128)
'''
-Converts each word (integer ID) into a dense vector (word embedding).
-input_dim=10000: only the top 10k most common words are considered.
-output_dim=128: each word will be represented as a 128-dimensional vector.
(Word vectors are often referred to as "embeddings".)'''
#Bidirectional(LSTM(64))
'''
-This is your Bi-LSTM layer, the heart of your model!
-It reads the sentence from both directions (forward and backward).
-LSTM(64) gives you 64 hidden units in each direction, so 128 total features.
(Context understanding + sentiment analysis (both forward and backward).)'''
#Dropout(0.5)
'''
-Randomly drops 50% of neurons during training to prevent overfitting.
-Helps generalize better to unseen data.
(Regularization to prevent overfitting)'''
#Dense(32, activation='relu')
'''
-A regular fully connected layer with 32 neurons.
-ReLU helps learn non-linear patterns.
-Think of this as your feature combiner before output.
(Combining features to learn complex patterns)'''
#Dense(1, activation='sigmoid')
'''
-The output layer.
-Returns a single number between 0 and 1 → representing probability of spam.
-Perfect for binary classification like spam vs ham.
(Binary classification output layer)
'''
'''
Text → Tokenized → Padded → Embedding → Bi-LSTM → Dropout → Dense → Output (0 or 1)
'''

'\nText → Tokenized → Padded → Embedding → Bi-LSTM → Dropout → Dense → Output (0 or 1)\n'

#Compile and Training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
es=EarlyStopping(patience=5,restore_best_weights=True,monitor='val_loss')

In [ ]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=[
                  BinaryAccuracy(name='accuracy'),
                  Precision(name='precision'),
                  Recall(name='recall'),
                  AUC(name='auc')
              ])
history=model.fit(X_train,y_train,
                  validation_data=(X_test,y_test),
                  epochs=30,
                  batch_size=32,callbacks=[es])


Epoch 1/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.8598 - loss: 0.3516 - val_accuracy: 0.9777 - val_loss: 0.0786
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9870 - loss: 0.0464 - val_accuracy: 0.9813 - val_loss: 0.0668
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9884 - loss: 0.0417 - val_accuracy: 0.9858 - val_loss: 0.0621
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9973 - loss: 0.0140 - val_accuracy: 0.9804 - val_loss: 0.0745
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9987 - loss: 0.0053 - val_accuracy: 0.9849 - val_loss: 0.0661
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9996 - loss: 0.0021 - val_accuracy: 0.9866 - val_loss: 0.0692
Epoch 7/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9995 - loss: 0.0022 - val_accuracy: 0.9858 - val_loss: 0.0792
Epoch 8/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 1.0000 - loss: 8.8412e-04 - val

In [ ]:
#Model.compile->tells the model how to be trained
'''
loss: tells the model how wrong it is — and guides learning by minimizing this loss.
optimizer: Adam is an optimization algorithm that updates weights efficiently
metrics: This tells Keras to track accuracy during training and validation.
'''
#Model.fit->fits the model to the data,starts the training process
'''
-X_train, y_train: The training data used to teach the model.
-validation_data=(X_test, y_test): After each epoch, the model evaluates on this test set to see how it's generalizing.
-epochs=5: The model will go through the entire dataset 5 times.
-batch_size=32: Model updates its weights every 32 samples.This balances speed and stability.
'''
'''---'''


'---'

#Evaluation and Prediction

In [ ]:
y_scores = model.predict(X_test).ravel()
y_pred = (y_scores > 0.5).astype(int)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
metrics_payload = {
    'model_name': 'Attention BiLSTM',
    'status': 'ready',
    'message': 'Metrics exported from Google Colab training run.',
    'artifacts': {
        'model_file': 'spam_ham_classifier_attention_bilstm.keras',
        'tokenizer_file': 'tokenizer.pkl'
    },
    'metrics': {
        'accuracy': round(float((y_test == y_pred).mean()), 4),
        'precision': round(float(tf.keras.metrics.Precision()(y_test, y_pred).numpy()), 4),
        'recall': round(float(tf.keras.metrics.Recall()(y_test, y_pred).numpy()), 4),
        'f1_score': round(float(f1_score(y_test, y_pred)), 4),
        'auc': round(float(tf.keras.metrics.AUC()(y_test, y_scores).numpy()), 4)
    },
    'confusion_matrix': {
        'true_positive': int(tp),
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn)
    },
    'training': {
        'epochs': len(history.history['loss']),
        'best_epoch': int(np.argmin(history.history['val_loss'])) + 1,
        'batch_size': 32,
        'validation_split': 0.2
    }
}

with open('model_metrics.json', 'w') as f:
    json.dump(metrics_payload, f, indent=2)

36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       956
           1       0.97      0.93      0.95       167

    accuracy                           0.99      1123
   macro avg       0.98      0.96      0.97      1123
weighted avg       0.99      0.99      0.99      1123



#making predictions on new text

In [ ]:
'''def predict_spam(text):
  cleaned=clean_text(text)
  seq=tokenizer.texts_to_sequences([cleaned])
  padded_seq=pad_sequences(seq,maxlen=100,padding='post',truncating='post')
  pred=model.predict(padded_seq)[0][0]
  return "Spam" if pred>0.5 else "Ham"
'''

'def predict_spam(text):\n  cleaned=clean_text(text)\n  seq=tokenizer.texts_to_sequences([cleaned])\n  padded_seq=pad_sequences(seq,maxlen=100,padding=\'post\',truncating=\'post\')\n  pred=model.predict(padded_seq)[0][0]\n  return "Spam" if pred>0.5 else "Ham"\n'

In [ ]:
'''
print(predict_spam("Congratulations! You won a free trip to the beach."))
'''

'\nprint(predict_spam("Congratulations! You won a free trip to the beach."))\n'

In [ ]:
model.save("spam_ham_classifier_attention_bilstm.keras")


import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)



In [ ]:
from google.colab import files
files.download("spam_ham_classifier_attention_bilstm.keras")
files.download("tokenizer.pkl")
files.download("model_metrics.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Attention BiLSTM is now defined above as the main `model` used for training and export.

In [4]:
print(model.summary())

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 100, 128)  │    640,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 100, 128)  │     98,816 │ embedding_1[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_1         │ (None, 100, 128)  │          0 │ bidirectional_1[… │
│ (Attention)         │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ attention_1[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      4,128 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 742,977 (2.83 MB)

 Trainable params: 742,977 (2.83 MB)

 Non-trainable params: 0 (0.00 B)

None
